# FilmScope Support Assistant: An Evaluated RAG System

**AI Engineering portfolio project**  
**Repository:** [github.com/enterlina/filmoskop-rag](https://github.com/enterlina/filmoskop-rag)

This notebook documents the design, evaluation, and iteration of a retrieval-augmented generation system for a fictional streaming service. The goal was not merely to produce plausible answers, but to build a reproducible pipeline, measure each component, diagnose failure modes, and justify the final engineering choices.

The final system keeps embeddings and retrieval local, uses routed BM25+dense retrieval and a cross-encoder for reranking, and calls an OpenAI-compatible NVIDIA endpoint only for answer generation and evaluation. All long-running stages use versioned disk checkpoints.

## 1. Problem and evaluation set

The knowledge base contains support articles in plain text, HTML, and Markdown. The evaluation set contains 66 questions in four groups:

- **exact** — explicit product terms, error codes, or limits;
- **semantic** — the user describes a problem without copying article wording;
- **multi** — the answer must combine facts from multiple articles;
- **gap** — the knowledge base genuinely contains no answer.

Retrieval metrics exclude gap questions. Generation evaluation includes them because correct refusal behavior is a product requirement. The project targets `recall@5 ≥ 0.85` and end-to-end `correctness ≥ 0.85`.

### Project objectives

1. Audit and clean the corpus, including encoding repair and removal of archive, service, and duplicate content.
2. Build a reproducible retrieval pipeline and compare dense search, BM25, reranking, Query Expansion, and HyDE.
3. Compare Nemotron and GPT-OSS 20B as grounded answer generators.
4. Implement deterministic abstention for questions that cannot be answered from the knowledge base.
5. Diagnose failures at retrieval and generation level and document the final engineering decision.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "filmoskop_kb.json").exists():
    candidate = PROJECT_DIR / "ypraktikum_task2" / "filmoskop_project"
    if (candidate / "filmoskop_kb.json").exists():
        PROJECT_DIR = candidate
CHECKPOINT_DIR = PROJECT_DIR / "filmoskop_checkpoints"
data = json.loads((PROJECT_DIR / "filmoskop_kb.json").read_text(encoding="utf-8"))
print(f"Documents: {len(data['docs'])} | Questions: {len(data['queries'])}")
print(Counter(q['kind'] for q in data['queries']))

## 2. System design and engineering rationale

1. **Corpus audit and cleaning.** HTML is parsed, mojibake is repaired line by line, and Unicode is normalized. Twenty-one archive pages and thirteen service documents are filtered by explicit title markers. Near-duplicates are detected from normalized body content at a 0.92 similarity threshold; the newest record is retained. No gold-relevant document is removed.
2. **Paragraph-aware chunking.** Articles are split on paragraph boundaries with the title repeated in every chunk. This preserves context while avoiding arbitrary sentence cuts.
3. **Routed hybrid retrieval.** `intfloat/multilingual-e5-base` provides exact cosine search. Queries containing codes, versions, numbers, or quoted identifiers additionally use BM25; dense and lexical lists are fused with RRF. Semantic requests stay on the dense route to avoid lexical noise.
4. **Cross-encoder reranking.** `BAAI/bge-reranker-v2-m3` reranks the top 20 candidates. It runs on CPU to avoid contention with Apple unified memory.
5. **Grounded generation and abstention.** The generator receives only the top five contexts. A text-only router selects separately calibrated dense-score thresholds for exact, semantic, and multi requests; insufficient evidence produces one canonical refusal.
6. **Evaluation.** Retrieval uses recall@K and MRR. Generation uses faithfulness, answer relevance, correctness, oracle correctness, and deterministic refusal checks.
7. **Checkpointing.** Embeddings, rankings, expansions, and every completed evaluation row are persisted atomically so interrupted runs resume instead of restarting.

## 3. Retrieval experiments and review iteration

The first iteration compared dense retrieval, cross-encoder reranking, Query Expansion, and HyDE. Reviewer feedback identified archive pages, service content, and near-duplicates still entering retrieval. Version 3 therefore cleans the corpus before indexing and routes identifier-heavy requests through BM25+dense RRF before the same cross-encoder. The table below reports the recomputed v3 retrieval metrics, not the older pre-cleaning scores.

In [ ]:
retrieval_v3 = pd.DataFrame({
    "Clean dense baseline": [0.758929, 0.655288],
    "Routed BM25 + dense": [0.767857, 0.652907],
    "Hybrid + cross-encoder": [0.946429, 0.829613],
}, index=["recall@5", "MRR"])
display(retrieval_v3.round(3))

retrieval_v3_by_type = pd.DataFrame({
    "recall@5": [1.000, 0.960, 0.875],
    "MRR": [1.000000, 0.718333, 0.843750],
}, index=["exact", "semantic", "multi"])
display(retrieval_v3_by_type)

**Final retrieval finding.** Corpus cleaning plus routed candidate generation and cross-encoder reranking raised recall@5 to 0.946 and MRR to 0.830. No archive page remained in any top-five result. Exact-query recall@5 reached 1.00, while multi remained the hardest slice at 0.875. BM25/RRF improved the candidate pool for identifiers, but most of the final gain came from the cross-encoder.

The earlier ablation remains useful as a negative result: Query Expansion did not improve multi or semantic recall@5 and slightly reduced exact-query MRR, while HyDE reduced performance in every group. The likely cause was semantic drift. These pre-cleaning ablation results are retained for experiment history but are not presented as final v3 metrics.

In [ ]:
ablation_path = CHECKPOINT_DIR / "retrieval_ablation_results.json"
if ablation_path.exists():
    ablation = json.loads(ablation_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(ablation["overall"]))
else:
    print("Independent Query Expansion / HyDE ablation has not been run yet.")

## 4. Generator A/B test (pre-cleaning baseline)

Before the v3 corpus change, two hosted generators were evaluated on exactly the same retrieved contexts:

- `nvidia/nemotron-3.5-lightning-30b-a3b`;
- `openai/gpt-oss-20b`.

Oracle correctness repeats generation with gold documents. It separates generator limitations from retrieval failures: if oracle correctness is high but end-to-end correctness is lower, retrieval is the remaining bottleneck. These scores justify the generator choice, but they must not be reported as post-cleaning v3 generation scores.

In [ ]:
generation_results = pd.DataFrame({
    "Nemotron": [0.500000, 0.530303, 0.500000, 0.515152],
    "GPT-OSS 20B": [0.803030, 0.909091, 0.787879, 0.909091],
}, index=["faithfulness", "answer relevance", "correctness", "oracle correctness"])
display(generation_results.round(3))
ax = generation_results.plot(kind="bar", figsize=(9, 4), ylim=(0, 1), rot=20)
ax.set_ylabel("Score")
ax.set_title("Generator comparison on 66 support questions")
ax.grid(axis="y", alpha=.25)
plt.tight_layout()

### GPT-OSS performance by question type

| Type | Faithfulness | Relevance | Correctness | Oracle correctness |
|---|---:|---:|---:|---:|
| exact | 0.933 | 0.933 | 0.867 | 0.933 |
| semantic | 0.680 | 0.840 | 0.720 | 0.880 |
| multi | 0.813 | 1.000 | 0.750 | 0.875 |
| gap | 0.900 | 0.900 | 0.900 | 1.000 |

GPT-OSS substantially outperformed Nemotron, especially on semantic and multi-document questions. Its 0.909 oracle correctness shows that it usually answers correctly when supplied with the right evidence. The 0.121 gap between oracle and end-to-end correctness points to retrieval and context selection as the next optimization target.

**Evaluation caveat:** GPT-OSS also acted as the LLM judge, which can introduce self-preference. Final claims should therefore be supported by manual review or an independent judge.

## 5. Failure analysis and fixes

The experiments revealed several concrete failure modes and led to explicit fixes:

- **Archived articles outranked current guidance — fixed in v3.** Explicit archive markers are filtered before chunking; 21 archive pages were removed and none remains in top-five retrieval.
- **Service pages and content duplicates added noise — fixed in v3.** Thirteen service documents and seven near-duplicates were removed with an auditable exclusion table.
- **Nemotron over-refused.** It frequently emitted the fallback even when the relevant article was ranked first. Low oracle correctness confirmed that this was a generator problem, not only retrieval.
- **Multi-document recall remains the weakest retrieval slice.** It improved to 0.875 recall@5, but some questions still miss the second required article.
- **Query Expansion and HyDE caused semantic drift.** Their combined configuration underperformed the simpler reranked pipeline.
- **Hosted APIs sometimes returned incomplete JSON.** Atomic checkpoints, validation, retries, and marked fallbacks prevented loss of completed work.

## 6. Final conclusion and selected configuration

This project delivers an end-to-end RAG prototype for the FilmScope support knowledge base: HTML cleanup, line-level mojibake repair, paragraph-aware chunking, embedding, retrieval, reranking, grounded generation, abstention, and automated evaluation. Atomic checkpoints make every expensive stage resumable after interruption.

The review iteration added explicit archive/service filtering and content-based near-duplicate removal. It removed 21 archive pages, 13 service documents, and 7 duplicates without removing any gold-relevant document. The final retrieval pipeline is **paragraph-aware multilingual E5 + routed BM25/RRF for exact identifiers → top-20 unique documents → BGE cross-encoder reranking → top-5 contexts**. On the cleaned corpus it achieved `recall@5 = 0.9464` and `MRR = 0.8296`, versus `0.7589` and `0.6553` for the clean dense baseline. Recall@5 by type was 1.00 exact, 0.96 semantic, and 0.875 multi.

GPT-OSS 20B remains the selected generator after substantially outperforming Nemotron in the original A/B test. Because corpus cleaning changes retrieved contexts, those earlier generation scores are retained as historical results only; v3 generation and judge metrics are written to a new checkpoint and must be recomputed before a final correctness claim.

### Final parameters

| Component | Selected value |
|---|---|
| Embedding model | `intfloat/multilingual-e5-base` |
| Indexing | Paragraph-aware chunks, normalized embeddings |
| Candidate retrieval | Routed BM25+dense RRF for exact identifiers; dense otherwise; top 20 |
| Reranker | `BAAI/bge-reranker-v2-m3`, CPU, batch size 16 |
| Generator context | Top 5 reranked documents |
| Generator | `openai/gpt-oss-20b` through NVIDIA API |
| Generation settings | `temperature=0`, `max_tokens=2048`, `reasoning_effort=low` |
| LLM judge | `openai/gpt-oss-20b`, JSON output, `temperature=0` |
| Abstention | Canonical refusal; route-specific 5th-percentile dense thresholds for exact/semantic/multi |
| Reproducibility | `seed=42`, atomic JSON/NPZ checkpoints, `CHECKPOINT_VERSION=3` |

The retrieval target (`recall@5 ≥ 0.85`) is exceeded. End-to-end correctness must now be recomputed for v3 against the reviewer target of `≥ 0.85`; the old 0.7879 score cannot be presented as a result of the changed pipeline. Production work would still require threshold calibration on a dedicated validation split and evaluation with an independent judge.

## 7. Reproducibility and security

The full experimental implementation is available in `filmoskop_rag_project.ipynb`. This portfolio notebook intentionally contains no credentials. API keys are entered through a hidden prompt or environment variables and are never written to the notebook.

Checkpoint artifacts include embeddings, baseline and experimental rankings, per-query generator evaluations, query expansions, and ablation results. Model names, preprocessing version, and relevant settings are part of cache keys, preventing stale or pre-mojibake-fix results from being mixed across configurations.